# BasketIQ — Step 1: Data Loading & Cleaning

**Dataset:** Online Retail Dataset (UCI Machine Learning Repository)

**Goal of this notebook:** load the raw Excel file, understand its structure, and clean it so it's ready for analysis in later steps (RFM segmentation, market basket analysis, etc.).

### Before running
1. Download `Online Retail.xlsx` from https://archive.ics.uci.edu/dataset/352/online+retail
2. Place it inside `BasketIQ/data/raw/`
3. Run the cells below in order.

## 1. Import libraries

We only need a few libraries for this step:
- `pandas` — to load and manipulate the tabular data
- `numpy` — for numeric operations (used later for cleaning checks)
- `pathlib.Path` — a clean, OS-independent way to build file paths (works the same on Windows/Mac/Linux)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Show all columns when printing a DataFrame (the dataset has 8 columns, so this keeps things readable)
pd.set_option('display.max_columns', None)

## 2. Load the raw data

The UCI file is an Excel file (`.xlsx`), so we use `pd.read_excel`.
We point to `data/raw/` because that folder is reserved for the **original, untouched** data — we never edit files there by hand, so we can always go back to the source if something goes wrong.

In [2]:
# Build the path to the raw file relative to this notebook's location
RAW_DATA_PATH = Path('..') / 'data' / 'raw' / 'Online Retail.xlsx'

# Load the Excel file into a DataFrame (this can take ~30-60 seconds, the file has 500k+ rows)
df = pd.read_excel(RAW_DATA_PATH)

# Show the first 5 rows so we can eyeball the structure
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 3. First look at the data

Before cleaning anything, we need to understand what we're working with:
- How many rows/columns?
- What are the data types?
- Where are the missing values?
- Are there obviously invalid values (e.g. negative prices)?

In [3]:
# .shape returns (number_of_rows, number_of_columns)
print('Shape (rows, columns):', df.shape)

# .info() shows column names, data types, and how many non-missing values each column has
df.info()

Shape (rows, columns): (541909, 8)
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [4]:
# .isnull().sum() counts how many missing (NaN) values are in each column
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [5]:
# .describe() gives summary statistics (min, max, mean, etc.) for numeric columns
# This helps us spot problems like negative Quantity or UnitPrice = 0
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


## 4. Clean the data

Based on the checks above, the Online Retail dataset typically has these issues:
1. **Missing `CustomerID`** — rows with no customer can't be used for customer-level analysis (like RFM), so we remove them.
2. **Cancelled orders** — invoices starting with the letter `C` (e.g. `C536379`) are cancellations, not real purchases. We remove them.
3. **Negative or zero `Quantity`** — can indicate returns or data errors. We keep only positive quantities.
4. **Negative or zero `UnitPrice`** — a price of 0 or less doesn't represent a real sale. We remove these rows.
5. **Wrong data types** — `CustomerID` loads as a float (e.g. 17850.0) because of the missing values; once missing values are gone we convert it to an integer, and `InvoiceDate` should be a proper datetime.

We keep a copy of the original `df` untouched and build a new cleaned DataFrame called `df_clean`, so we can always compare before/after.

In [6]:
# Start from a copy so the original df is never modified
df_clean = df.copy()

# Step 1: drop rows where CustomerID is missing
df_clean = df_clean.dropna(subset=['CustomerID'])

# Step 2: remove cancelled orders
# InvoiceNo is normally a string; cancelled invoices start with 'C'
# .astype(str) makes sure we can safely check the first character even if pandas read it as a number
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# Step 3: keep only positive quantities (removes returns / data errors)
df_clean = df_clean[df_clean['Quantity'] > 0]

# Step 4: keep only positive unit prices
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# Step 5: fix data types
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)   # whole numbers, no need for decimals
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])  # ensure proper datetime type

# Step 6: remove exact duplicate rows (same invoice, product, customer, etc. repeated identically)
df_clean = df_clean.drop_duplicates()

# Reset the row index so it's clean and sequential (0, 1, 2, ...) after all the filtering above
df_clean = df_clean.reset_index(drop=True)

print('Rows before cleaning:', len(df))
print('Rows after cleaning: ', len(df_clean))

Rows before cleaning: 541909
Rows after cleaning:  392692


## 5. Add a helper column: TotalPrice

Most of our future analysis (RFM, revenue trends, basket value) needs the total value of each line item, not just the unit price. We calculate it once here so we don't repeat this logic later:

`TotalPrice = Quantity x UnitPrice`

In [7]:
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


## 6. Sanity check the cleaned data

Quick final checks to confirm the cleaning worked as expected before saving.

In [8]:
# There should be no missing values left in any column
print('Remaining missing values per column:')
print(df_clean.isnull().sum())

print('\nData types:')
print(df_clean.dtypes)

print('\nDate range:', df_clean['InvoiceDate'].min(), 'to', df_clean['InvoiceDate'].max())
print('Number of unique customers:', df_clean['CustomerID'].nunique())
print('Number of unique invoices:', df_clean['InvoiceNo'].nunique())
print('Number of unique countries:', df_clean['Country'].nunique())

Remaining missing values per column:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
dtype: int64

Data types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              int64
Country                   str
TotalPrice            float64
dtype: object

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Number of unique customers: 4338
Number of unique invoices: 18532
Number of unique countries: 37


## 7. Save the cleaned dataset

We save the cleaned data to `data/processed/` as a CSV. This keeps the raw file untouched in `data/raw/` and gives us a fast, reusable starting point for the next notebook, instead of re-running the cleaning steps every time.

In [9]:
PROCESSED_DATA_PATH = Path('..') / 'data' / 'processed' / 'online_retail_clean.csv'

# index=False avoids writing the pandas row index as an extra column in the CSV
df_clean.to_csv(PROCESSED_DATA_PATH, index=False)

print(f'Saved cleaned dataset to: {PROCESSED_DATA_PATH.resolve()}')

Saved cleaned dataset to: D:\New folder\BasketIQ\data\processed\online_retail_clean.csv
